In [ ]:
# ---- Polynomial Expansion ----
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_scaled)
print(f"\nPolynomial expansion: {X_scaled.shape[1]} -> {X_poly.shape[1]} features")

In [ ]:
# ---- NEW: Apply SMOTE to the Polynomial Features ----
# This is the main engine for jumping to ~95% accuracy!
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_poly, y)
print(f"Resampled dataset shape: {X_resampled.shape}, Target distribution:\n{y_resampled.value_counts()}")

In [ ]:
# ---- Corrected Train/Test Split ----
# We now split X_resampled instead of X_scaled
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)
print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")

def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan

    print(f"\n{model_name} Performance:")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-score : {f1:.4f}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["Not Potable", "Potable"]))

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": roc_auc}